# Demo notebook for testing all functionality

In [6]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append('../')
import numpy as np

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Set environment constraint params

In [7]:
ns_B = 8  # maxB should be on boundary (so we could always just sample the boundary...)
target_volavgB = 1.0  # tesla
ntheta_B = 16
nzeta_B = 16
len_B_field_out = ns_B * ntheta_B * nzeta_B
mirror_target = 1.35
eps_B = (mirror_target - 1.0) / (mirror_target + 1.0)
B_ub = target_volavgB * (1 + eps_B) * np.ones(len_B_field_out)  # upper bound, eq. 14
B_lb = target_volavgB * (1 - eps_B) * np.ones(len_B_field_out)  # lower bound, eq. 14

In [8]:
B_ub.shape

(2048,)

## Initial candidate generation (for nonlinear inequality constraints in the acquisition function)


## Nonlinear inequality constraints in the acquisition function  (BoTorch)


### Background
Acquisition function optimization in Botorch (in `optimize_acqf`) uses multi-start gradient ascent from multiple initial conditions. This approach helps deal with nonconvexities and local maxima in the response surface of acquisition functions.

### Default Initialization
- Botorch normally uses a default initialization heuristic.
- This is based on batch evaluation of the acquisition function at many random points.
- It then performs Boltzmann sampling on the normalized values.
- See [`gen_batch_initial_conditions`](https://github.com/pytorch/botorch/blob/main/botorch/optim/initializers.py#L50) for implementation details.

## Challenge with Nonlinear Constraints
Sampling from the feasible set becomes very difficult with nonlinear constraints. The default BoTorch initializer doesn't support this scenario.

## Options for Handling Nonlinear Constraints

### Option 1: Provide Custom Initial Conditions
- Pass a set of initial conditions to perform multi-start ascent.
- Use the `batch_initial_conditions` argument in `optimize_acqf`.

### Option 2: Create a Custom Initializer
- Write own initializer callable with the same signature as `gen_batch_initial_conditions`.
- This should implement a strategy for generating initial conditions that satisfy the constraints.
- Pass this to `optimize_acqf` using the `ic_generator` keyword argument.

## Choosing an Approach
The best option depends on specific problem and constraints:
- If can easily generate feasible points, Option 1 might be simpler.
- For a more general or adaptive approach, Option 2 could be more suitable.

### Rejection sampling

In [10]:
from src.tracing_example import get_B_field

In [15]:
# Reference: x0 = np.array([0.  , 0.  , 2.38, 0.  , 0.  , 0.  , 2.38, 0.  ])
B_lower_constraint = lambda x, cache: B_lb - get_B_field(x, cache)
B_upper_constraint = lambda x, cache: get_B_field(x, cache) - B_ub
# Number of Fourier modes for optimization
max_mode = 1
d = 4 * max_mode**2 + 4 * max_mode
lower = -1e1
upper = 1e1

def generate_random_samples_under_nonlinear_constraint(n_samples:int, d:int, upper:float, lower:float):
    samples = []
    cache = {}
    for _ in range(n_samples):
        x = np.random.uniform(upper, lower, size=(d,))
        # TODO: @misha - stop VMEC from generating crap everytime it is called
        if np.all(B_lower_constraint(x, cache) <= 0) and np.all(B_upper_constraint(x, cache) <= 0):
            samples.append(x) # Valid sample
    return np.array(samples)

In [16]:
samples = generate_random_samples_under_nonlinear_constraint(10, d, upper, lower)